# 🦇 Creando una Aplicación de Chat con MLflow y la API de OpenAI en Databricks

## ¿Qué vamos a construir en este notebook?

En este notebook vamos a crear, paso a paso, un **chatbot** (una aplicación que conversa) y lo vamos a **desplegar** (poner en producción) dentro de Databricks para que cualquier persona o aplicación pueda usarlo a través de internet mediante una *URL*.

El recorrido completo (este es el **flujo de trabajo** típico de un proyecto de GenAI en Databricks) es:

1. **Instalar librerías** (`openai`, `mlflow`).
2. **Conectarnos a un modelo de lenguaje (LLM)** ya alojado en Databricks usando el cliente de OpenAI.
3. **Hacer una primera petición de prueba** para entender cómo se habla con un LLM.
4. **Empaquetar nuestra lógica dentro de una clase de Python** compatible con MLflow (`pyfunc.PythonModel`).
5. **Guardar (`save`)** el modelo en disco.
6. **Cargar (`load`)** el modelo guardado para probarlo.
7. **Registrar (`log`)** el modelo como un *experimento* de MLflow.
8. **Registrarlo (`register`)** en **Unity Catalog** (el catálogo de gobierno de datos de Databricks).
9. **Desplegarlo (`deploy`)** como un *endpoint* de **Mosaic AI Model Serving** (una API en tiempo real).

> 📚 **Nota para el aprendizaje**: a lo largo del notebook encontrarás celdas de texto (como esta) que explican el *qué* y el *por qué*, y celdas de código que ejecutan acciones. Lee siempre el texto antes de ejecutar el código de debajo.

---

### 🧠 Conceptos clave antes de empezar

| Concepto | ¿Qué es? |
|----------|----------|
| **LLM** (*Large Language Model*) | Modelo de lenguaje grande, como Claude o GPT, que genera texto. |
| **MLflow** | Plataforma de código abierto (integrada en Databricks) para gestionar el ciclo de vida de los modelos: registrar, versionar, empaquetar y desplegar. |
| **Mosaic AI Model Serving** | El servicio de Databricks que pone un modelo detrás de una API REST en tiempo real (un *endpoint*). |
| **Unity Catalog (UC)** | El sistema de gobierno de Databricks. Organiza todo en `catálogo.esquema.objeto` y controla permisos. Aquí registramos la versión "oficial" del modelo. |
| **Endpoint** | La *URL* a la que se envían peticiones para obtener respuestas del modelo. |
| **Serverless / Scale to Zero** | El servidor que aloja el modelo se apaga solo cuando no se usa (coste 0) y se enciende cuando llega una petición. Lo explicamos en detalle al final. |

![llm_chatbot_model_serving](./Assets/llm_chatbot_model_serving.png)

## 1️⃣ Instalación de utilidades y librerías

Antes de programar necesitamos instalar dos librerías de Python en el *cluster* (el ordenador remoto donde se ejecuta el notebook):

- **`openai`**: la librería oficial de OpenAI. Aunque se llame "openai", Databricks expone sus modelos (Claude, Llama, etc.) usando el **mismo formato/protocolo que OpenAI**, así que reutilizamos este cliente para hablar con modelos de Databricks. Es un estándar de la industria.
- **`mlflow==3.0.1`**: fijamos la versión exacta de MLflow (`==` significa "exactamente esta versión") para que el notebook sea **reproducible** y no se rompa si sale una versión nueva con cambios.

### 🔧 El comando mágico `%pip`

`%pip` es un *magic command* (comando mágico) de los notebooks. Las celdas que empiezan por `%` no son Python normal, son instrucciones especiales para el entorno del notebook:

| Magic command | Para qué sirve |
|---------------|----------------|
| `%pip install <paquete>` | Instala librerías de Python en **todos los nodos** del cluster. |
| `%sql` | Ejecuta la celda como SQL en lugar de Python. |
| `%md` | Convierte la celda en texto Markdown. |
| `%sh` | Ejecuta comandos de shell (Linux) en el nodo *driver*. |
| `%run ./otro_notebook` | Ejecuta otro notebook (útil para reutilizar funciones). |
| `%fs` | Comandos del sistema de ficheros (DBFS). |

> 💡 **Para la certificación**: recuerda que `%pip` instala a nivel de *sesión del notebook*. Para instalar librerías de forma permanente en el cluster se usan las *Cluster Libraries* desde la interfaz, o ficheros `requirements.txt` / *init scripts*.

In [ ]:
%pip install openai mlflow==3.0.1

## 2️⃣ Reiniciando el entorno de Python

### ¿Por qué reiniciar?

Cuando instalas librerías nuevas con `%pip install`, Python **ya tenía cargada en memoria** la versión anterior (o ninguna). Para que el notebook "vea" las librerías recién instaladas hay que **reiniciar el intérprete de Python**.

`dbutils.library.restartPython()` hace justo eso: reinicia el proceso de Python del notebook **sin perder la conexión con el cluster**.

### ⚠️ Efecto importante

Al reiniciar, **se borran todas las variables** que tuvieras en memoria (cualquier `x = ...` que hubieras ejecutado antes). Por eso la regla de oro es:

> **Instala librerías → reinicia Python → y SOLO DESPUÉS empieza a programar.**

### 🧰 ¿Qué es `dbutils`?

`dbutils` (*Databricks Utilities*) es un conjunto de herramientas que Databricks te da gratis dentro de cualquier notebook. Las familias más importantes:

| Módulo | Para qué sirve | Ejemplo |
|--------|----------------|---------|
| `dbutils.fs` | Manejar ficheros (DBFS, volúmenes) | `dbutils.fs.ls("/")` |
| `dbutils.widgets` | Crear parámetros/controles en el notebook | `dbutils.widgets.text("nombre", "valor")` |
| `dbutils.secrets` | Leer contraseñas/tokens guardados de forma segura | `dbutils.secrets.get("scope", "clave")` |
| `dbutils.library` | Gestionar librerías | `dbutils.library.restartPython()` |
| `dbutils.notebook` | Ejecutar/encadenar otros notebooks | `dbutils.notebook.run(...)` |

> 💡 **Truco**: escribe `dbutils.help()` en una celda para ver toda la documentación.

In [ ]:
dbutils.library.restartPython()

## 3️⃣ Creando el cliente de OpenAI (apuntando a Databricks)

### ¿Qué es un "cliente"?

Un **cliente** es un objeto de Python que sabe cómo comunicarse con un servidor. En lugar de escribir tú las peticiones HTTP a mano, el cliente te da métodos cómodos como `client.chat.completions.create(...)`.

Para crearlo necesitamos dos datos:

| Parámetro | Qué es | Dónde se consigue |
|-----------|--------|-------------------|
| `api_key` | Tu **token de acceso personal (PAT)**. Funciona como una contraseña que identifica quién hace la petición. | En Databricks: *Settings → Developer → Access Tokens → Generate new token*. |
| `base_url` | La **dirección** del servicio de *serving* de tu workspace. Termina en `/serving-endpoints`. | Es `https://<tu-workspace>.cloud.databricks.com/serving-endpoints`. La parte del workspace está en la URL del navegador. |

### 🔐 MUY IMPORTANTE: no pongas tokens en el código

En este ejemplo el token está escrito directamente (`"YOUR_DATABRICKS_ACCESS_TOKEN"`) **solo por simplicidad didáctica**. En un proyecto real **NUNCA** se escribe una contraseña en el código, porque quedaría visible para todo el que vea el notebook o el repositorio de Git.

La forma correcta y segura es usar **Databricks Secrets**:

```python
api_key = dbutils.secrets.get(scope="mi-scope", key="databricks-token")
```

O, dentro de un notebook de Databricks, puedes obtener el host y un token automático del contexto:

```python
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = "https://" + ctx.tags().get("browserHostName").get()
token = ctx.apiToken().get()
```

> 💡 **Para la certificación**: saber que los *secrets scopes* son la forma recomendada de gestionar credenciales (tokens, claves de API) es un punto típico de examen.

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key = "YOUR_DATABRICKS_ACCESS_TOKEN",
    base_url = "YOUR_DATABRICKS_WORKSPACE_HOSTNAME/serving-endpoints"
)

## 4️⃣ Enviando una petición de chat simple

Aquí hacemos nuestra **primera conversación** con el modelo. Es importante entender la estructura de una petición a un LLM porque se repite siempre.

### 🗣️ Los "roles" de los mensajes

Una conversación con un LLM se envía como una **lista de mensajes**, y cada mensaje tiene un *rol*:

| Rol | Quién habla | Para qué sirve |
|-----|-------------|----------------|
| `system` | El desarrollador (tú) | Da **instrucciones de comportamiento**: la personalidad, las reglas, el tono. Aquí le decimos *"Eres Batman, el protector de Gotham"*. |
| `user` | El usuario final | Es la **pregunta o mensaje** de la persona. |
| `assistant` | El propio modelo | Son las **respuestas anteriores** del modelo (se incluyen para que recuerde el contexto en una conversación larga). |

### 🎛️ Parámetros importantes de la petición

| Parámetro | Qué controla | Valores típicos |
|-----------|--------------|-----------------|
| `model` | Qué LLM usar (el nombre del endpoint en Databricks) | `databricks-claude-sonnet-4-5`, `databricks-meta-llama-3-1-70b-instruct`, ... |
| `messages` | La conversación (lista de roles) | — |
| `temperature` | La **creatividad**: 0 = respuestas deterministas y precisas; 1 = más variadas y creativas | 0.0 – 1.0 |
| `max_tokens` | Longitud máxima de la respuesta (en *tokens* ≈ trozos de palabra) | 256, 512, 1000... |
| `top_p` | Alternativa a `temperature` (muestreo por probabilidad acumulada) | 0.0 – 1.0 |
| `stream` | Si `True`, devuelve la respuesta palabra a palabra (como ChatGPT escribiendo) | `True` / `False` |

### 📦 Cómo leer la respuesta

La respuesta es un objeto. El texto que nos interesa está en:
`completion.choices[0].message.content`

- `choices` es una lista (puede haber varias respuestas posibles); `[0]` toma la primera.
- `message.content` es el texto generado.

> 🧩 **Foundation Model APIs**: en Databricks estos modelos preinstalados (`databricks-...`) se facturan por *pay-per-token* (pagas por uso) y no necesitas desplegar nada para usarlos. Son los *Foundation Model APIs*.

In [ ]:
# OpenAI Request
completion = client.chat.completions.create(
    model="databricks-claude-sonnet-4-5",
    messages=[
        {
            "role":"system",
            "content": [
                {
                    "type": "text", "text": "You are Batman, the protector of Gotham City"
                }
            ]

        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text", "text": "How's Gotham City doing today?"
                }
            ]
        }
    ]
)

# printing the response
print(completion.choices[0].message.content)

## 5️⃣ Empaquetando nuestra lógica en una clase de Python (modelo `pyfunc` de MLflow)

Esta es la celda **más importante** del notebook y donde aparece la **Programación Orientada a Objetos (POO)**. Vamos despacio.

### 🧱 ¿Qué es una clase? (POO explicada desde cero)

Imagina una clase como el **plano de una casa** 🏠. El plano no es una casa de verdad, pero describe cómo construir casas. Cuando usas el plano para construir una casa concreta, esa casa es un **objeto** (o *instancia*).

- **Clase** = el molde/plano/receta.
- **Objeto / instancia** = algo concreto creado a partir de la clase.
- **Atributos** = las características que guarda el objeto (variables dentro de él, p. ej. `self.model_name`).
- **Métodos** = las acciones que puede hacer el objeto (funciones dentro de la clase, p. ej. `predict`).

```python
class Perro:                  # el plano
    def __init__(self, nombre):
        self.nombre = nombre  # atributo
    def ladrar(self):         # método
        return f"{self.nombre} dice ¡guau!"

mi_perro = Perro("Toby")      # objeto/instancia
mi_perro.ladrar()             # -> "Toby dice ¡guau!"
```

### 🔑 Las piezas que verás en el código

| Pieza | Qué significa |
|-------|---------------|
| `class BasicChatBot(pyfunc.PythonModel):` | Creamos una clase llamada `BasicChatBot` que **hereda** de `PythonModel`. *Heredar* significa "tomar como base"; al heredar de `PythonModel`, MLflow ya sabe cómo guardar y servir nuestra clase. |
| `def __init__(self, model_name):` | El **constructor**. Es el método que se ejecuta automáticamente al crear el objeto. Sirve para inicializar los atributos. |
| `self` | Es **el propio objeto**. Siempre es el primer parámetro de los métodos. `self.model_name` guarda el dato *dentro* del objeto para usarlo en otros métodos. |
| `def predict(self, context, data):` | El método **obligatorio** que MLflow llama cuando alguien hace una petición al modelo servido. Su firma (`context`, `data`) la exige MLflow. Aquí va la lógica de inferencia. |
| `def chatCompletionsAPI(self, user_query):` | Un método **auxiliar** propio nuestro: contiene la llamada al LLM. `predict` lo invoca internamente. |

### 🔄 ¿Por qué meter todo en una clase? (el "por qué" del flujo)

Para **desplegar** un modelo en Databricks, MLflow necesita un objeto que siga un contrato estándar: tener un método `predict(context, data)`. Al envolver nuestra llamada al LLM dentro de una clase `PythonModel`, conseguimos que:

1. MLflow pueda **guardar** (serializar) toda la lógica.
2. El servidor de Mosaic AI pueda **cargarla** y llamar a `predict()` automáticamente cada vez que llega una petición HTTP.

Esto se llama un **modelo `pyfunc`** (*python function*): un formato genérico de MLflow que envuelve cualquier código Python para que se comporte como un modelo estándar. Es la base para servir LLMs, agentes o pipelines personalizados.

> ⚠️ **Detalle importante de producción**: en este ejemplo el `OpenAI(...)` se crea **dentro** de `chatCompletionsAPI`, así que el cliente se reconstruye en cada llamada. Está bien para aprender, pero fíjate en que el token vuelve a estar escrito a mano (otra vez, en real usarías `secrets`).

### 🔬 Profundizando: ¿qué es `pyfunc` exactamente?

`pyfunc` (*python function*) es el **formato universal de modelos de MLflow**. La idea de fondo es muy potente: **da igual con qué librería se creó el modelo** (scikit-learn, PyTorch, XGBoost, Hugging Face, o código tuyo a mano como nuestro chatbot) — MLflow lo "envuelve" para que **todos se usen exactamente igual**:

```python
modelo = mlflow.pyfunc.load_model(...)
modelo.predict(datos)   # ← SIEMPRE la misma interfaz, sea el modelo que sea
```

Esa uniformidad es justo lo que permite que **Mosaic AI Model Serving** despliegue cualquier modelo sin saber qué hay dentro: el endpoint solo necesita llamar a `.predict()`.

#### 🍨 Los "flavors" (sabores) de un modelo

Un modelo en MLflow puede guardarse en varios **flavors** a la vez. Un *flavor* es simplemente una "forma de cargar" el modelo:

| Flavor | Para qué tipo de modelo |
|--------|-------------------------|
| `mlflow.sklearn` | modelos de scikit-learn |
| `mlflow.pytorch` | redes de PyTorch |
| `mlflow.transformers` | modelos de Hugging Face |
| `mlflow.langchain` | cadenas/agentes de LangChain |
| **`mlflow.pyfunc`** | **el genérico — casi todos los modelos lo tienen también** |

Por eso `pyfunc` es el "denominador común": aunque guardes un modelo de sklearn, **también** podrás cargarlo como `pyfunc` y servirlo igual que cualquier otro. Es el flavor que usa el motor de *serving*.

#### ✌️ Dos formas de crear un modelo `pyfunc` propio

1. **Subclase de `PythonModel`** (lo que hacemos en este notebook): defines una **clase** con un método `predict()`. Es la opción cuando hay **lógica o estado** (un cliente, configuración, pasos previos...).
2. **Función suelta** (*python_function*): le pasas directamente una **función** de Python. Útil para transformaciones simples sin estado.

#### 🧰 Opciones útiles al hacer una subclase `PythonModel`

| Pieza | Qué hace | Cuándo usarla |
|-------|----------|---------------|
| `predict(self, context, model_input)` | **Obligatorio.** La inferencia (la respuesta del modelo). | Siempre. |
| `load_context(self, context)` | **Opcional pero clave.** Se ejecuta **una sola vez al arrancar** el modelo (al cargarlo o al encender el endpoint), **no** en cada petición. | Crear el cliente, abrir conexiones, cargar pesos o ficheros pesados **una vez** y reutilizarlos. |
| `context.artifacts` | Diccionario con rutas a **ficheros extra** que empaquetaste con el modelo (un `.json`, pesos, un índice de búsqueda...). | Acceder a esos ficheros dentro de `load_context`/`predict`. |
| `artifacts=` (en `log_model`/`save_model`) | Empaqueta ficheros auxiliares junto al modelo. | Modelos que necesitan datos de apoyo. |
| `pip_requirements=` / `extra_pip_requirements=` | Fija las **dependencias** (librerías) que el endpoint instalará. | Garantizar que en producción estén las mismas versiones. |
| `code_paths=` | Incluye módulos `.py` propios en el paquete. | Reutilizar código tuyo de otros ficheros. |

> 💡 **Mejora de producción que aplicaremos en el código de abajo**: en la versión inicial el cliente `OpenAI` se creaba **dentro de `predict`**, así que se reconstruía en *cada* petición (lento e ineficiente). Lo correcto es crearlo en **`load_context`**, que se ejecuta **una sola vez** al arrancar el endpoint, y reutilizarlo en todas las peticiones. Es exactamente el tipo de detalle que pueden preguntar en la certificación.

In [ ]:
import mlflow
from mlflow import pyfunc

# Definimos una CLASE que hereda de pyfunc.PythonModel.
# Heredar de PythonModel hace que MLflow sepa guardar y servir este objeto.
class BasicChatBot(pyfunc.PythonModel):

    # __init__ es el CONSTRUCTOR: se ejecuta al crear el objeto en Python.
    # 'self' es el propio objeto; guardamos el nombre del modelo como ATRIBUTO.
    # OJO: aquí NO creamos el cliente todavía (lo hacemos en load_context).
    def __init__(self, model_name: str):
        self.model_name = model_name
        self.client = None  # se rellenará en load_context

    # load_context() se ejecuta UNA SOLA VEZ al cargar el modelo
    # (al hacer load_model o al arrancar el endpoint), NO en cada petición.
    # Es el sitio correcto para crear clientes/conexiones y reutilizarlos.
    def load_context(self, context):
        # En producción, lee el token con dbutils.secrets en lugar de escribirlo.
        self.client = OpenAI(
            api_key = "YOUR_DATABRICKS_ACCESS_TOKEN",
            base_url = "YOUR_DATABRICKS_WORKSPACE_HOSTNAME/serving-endpoints"
        )

    # Método AUXILIAR propio: contiene la llamada real al LLM.
    # Reutiliza self.client (creado una sola vez en load_context).
    def chatCompletionsAPI(self, user_query):
        response = self.client.chat.completions.create(
            model = self.model_name,   # usamos el atributo guardado en __init__
            messages = [
                {
                    "role": "system",  # instrucciones de comportamiento (la personalidad)
                    "content": [
                        {
                            "type": "text", "text": "You are Batman, the protector of Gotham City"
                        }
                    ]
                },
                {
                    "role": "user",    # el mensaje del usuario final
                    "content": [
                        {
                            "type": "text", "text": user_query
                        }
                    ]
                }
            ],
            temperature=0.7  # 0 = preciso/determinista, 1 = creativo
        )

        return response.choices[0].message.content

    # predict() es el método OBLIGATORIO que MLflow llama en el endpoint.
    # Su firma (context, data) la exige MLflow; NO la cambies.
    def predict(self, context, data):
        # Salvaguarda: si se llamó a predict sin pasar por load_context
        # (p. ej. al usar el objeto directamente en el notebook), creamos el cliente.
        if self.client is None:
            self.load_context(context)
        user_query = data["user_query"].iloc[0]  # 1er valor de la columna user_query
        gpt_response = self.chatCompletionsAPI(user_query)
        return gpt_response

## 6️⃣ Guardando nuestro modelo (`save_model`)

Ahora vamos a **guardar** el modelo en disco. Esto se hace en dos pasos: primero creamos una *instancia* de la clase, y luego la guardamos con MLflow.

### 🧬 La "firma" del modelo (`signature`) y el ejemplo de entrada

Antes de guardar definimos la **firma** (*signature*): un "contrato" que describe **qué entra y qué sale** del modelo.

- **`input_example`**: un ejemplo de los datos de entrada. Aquí, un `DataFrame` de pandas con una columna `user_query`.
- **`output_example`**: un ejemplo de la salida.
- **`infer_signature(input, output)`**: MLflow **deduce automáticamente** los tipos de las columnas a partir de los ejemplos y construye la firma.

¿Por qué importa la firma? Porque:
1. **Documenta** el modelo (cualquiera sabe cómo llamarlo).
2. **Valida** las peticiones en producción (si alguien manda datos con el formato equivocado, falla pronto y con un error claro).
3. Es **obligatoria** para registrar modelos en Unity Catalog.

### 💾 `mlflow.pyfunc.save_model(...)`

| Parámetro | Qué hace |
|-----------|----------|
| `path` | Carpeta local donde se guardarán los *artefactos* del modelo. |
| `python_model` | El objeto que queremos guardar (nuestra instancia de `BasicChatBot`). |
| `signature` | La firma que acabamos de inferir. |
| `input_example` | Se guarda junto al modelo como ejemplo de uso. |

### 🆚 `save_model` vs `log_model` (¡pregunta típica de examen!)

| | `save_model` | `log_model` |
|---|---|---|
| **Dónde guarda** | En una carpeta local que tú eliges | Dentro de un **run** (ejecución) de MLflow, con seguimiento de experimentos |
| **¿Hay tracking?** | No | Sí: queda registrado en la UI de *Experiments* |
| **Uso típico** | Pruebas rápidas | Lo habitual en proyectos reales |

> Lo más común en producción es usar directamente `mlflow.pyfunc.log_model(...)`, que guarda **y** registra en un solo paso. Aquí se hace separado para fines didácticos.

### 🔬 Profundizando: ¿qué es la `signature` (firma) y para qué sirve?

La **firma** es el **contrato de entrada y salida** del modelo: describe **qué columnas y tipos** espera recibir y **qué devuelve**. Es como la "etiqueta de ingredientes" 🏷️ del modelo: cualquiera (una persona o el endpoint) sabe cómo hablarle.

```python
input_example  = pd.DataFrame([{"user_query": "how is Gotham?"}])
output_example = pd.DataFrame([{"predictions": "Gotham is..."}])
signature = infer_signature(input_example, output_example)
```

`infer_signature` **mira los ejemplos y deduce los tipos automáticamente**. La firma resultante es un esquema parecido a esto:

```
inputs:  ['user_query': string]
outputs: ['predictions': string]
```

#### 🎯 ¿Para qué sirve? (3 razones)

1. **Validación en producción** 🛡️: si alguien envía al endpoint datos con el formato equivocado (una columna mal escrita, un número donde esperaba texto), MLflow **rechaza la petición con un error claro** en vez de fallar de forma rara dentro del modelo.
2. **Documentación** 📖: cualquiera que vea el modelo en la interfaz sabe exactamente cómo invocarlo.
3. **Requisito de Unity Catalog** ✅: para **registrar** un modelo en UC, la firma es **obligatoria**.

#### 🧱 Tipos de firma

| Tipo | Qué describe | Ejemplo de uso |
|------|--------------|----------------|
| **Column-based** (basada en columnas) | Datos tabulares (un `DataFrame`): nombres y tipos de columnas. | La de nuestro chatbot (`user_query`). |
| **Tensor-based** (basada en tensores) | Arrays/tensores: forma (*shape*) y `dtype`. | Imágenes, audio, deep learning. |
| **Con `params`** | Permite **parámetros de inferencia opcionales** (p. ej. `temperature`, `max_tokens`) que quien llama puede ajustar en cada petición. | Hacer el modelo configurable por petición. |

#### ⚠️ No confundas estos dos conceptos (cae en el examen)

| Concepto | Qué es |
|----------|--------|
| **`input_example`** | Un **dato de muestra** real. Sirve para *inferir* la firma y para que el endpoint tenga un ejemplo de prueba. |
| **`signature`** | El **esquema/contrato formal** (los tipos), derivado normalmente de ese ejemplo. |

> 💡 **Atajo habitual en producción**: no siempre defines la firma a mano. Si pasas `input_example=` a `log_model`/`save_model`, **MLflow infiere la firma automáticamente** a partir de ese ejemplo. Definirla explícitamente con `infer_signature` (como aquí) te da más control y es más claro para aprender.

In [ ]:
test_model = BasicChatBot(model_name="YOUR_MODEL_NAME") # e.g., "databricks-claude-sonnet-4-5"

In [ ]:
from mlflow.models import infer_signature
import pandas as pd
import mlflow

# Sample input
input_example = pd.DataFrame([
    {"user_query": "how is Gotham City today?"}
])

# Sample output (what your model actually returns)
output_example = pd.DataFrame([
    {
        "predictions": "Gotham City is facing cloudy weather with rising tensions downtown."
    }
])

# Infer full signature (input + output)
signature = infer_signature(input_example, output_example)

model_path = "basicchatbot"

mlflow.pyfunc.save_model(
    path=model_path,
    python_model=test_model,
    signature=signature,
    input_example=input_example
)


## 7️⃣ Cargando el modelo guardado (`load_model`)

`mlflow.pyfunc.load_model(model_path)` hace lo contrario de guardar: **reconstruye** el objeto del modelo a partir de los artefactos del disco.

¿Por qué cargarlo si ya lo tenemos en memoria? Porque al cargarlo desde el disco probamos **exactamente lo que se desplegará** en producción. Es una verificación de que el empaquetado funciona:

- Se carga como un objeto genérico `pyfunc`.
- Ya **no** llamas a `chatCompletionsAPI` ni a `__init__` directamente: solo tienes el método estándar `.predict(...)`, igual que lo usará el endpoint.

> 🔁 Este "guardar → cargar → predecir" es el patrón de prueba recomendado antes de registrar y desplegar. Si falla aquí, fallará en el endpoint.

In [ ]:
# Load our custom model from the local artifact store
loaded_pyfunc_model = mlflow.pyfunc.load_model(model_path)

## 8️⃣ Probando el modelo cargado

Le pasamos un `DataFrame` con la columna `user_query` (igual que la firma que definimos) y llamamos a `.predict(...)`.

Internamente ocurre esto:
1. MLflow llama a nuestro método `predict(self, context, data)`.
2. Nuestro `predict` extrae `data["user_query"].iloc[0]` (el primer valor de la columna).
3. Llama a `chatCompletionsAPI`, que contacta con el LLM.
4. Devuelve el texto de la respuesta.

> ✅ Si ves la respuesta de "Batman" impresa, ¡el empaquetado funciona! Ya podemos registrarlo y desplegarlo con confianza.

In [ ]:
model_input = pd.DataFrame([{"user_query": "hello how are you?"}])

model_response = loaded_pyfunc_model.predict(model_input)

print(model_response)

## 9️⃣ Registrando el modelo como un "run" de MLflow (`log`)

### 📒 ¿Qué es un *run* (ejecución) y un *experiment* (experimento)?

MLflow organiza el trabajo así:
- **Experiment** = una carpeta lógica que agrupa intentos relacionados (por ejemplo, "ChatBot Batman").
- **Run** = **una ejecución concreta** dentro de un experimento. Cada vez que entrenas/pruebas, creas un run con su propio identificador (`run_id`), sus parámetros, métricas y artefactos.

### 🧩 El bloque `with mlflow.start_run() as run:`

```python
with mlflow.start_run() as run:
    ...
```

`with` es un **gestor de contexto** de Python: abre algo al entrar y lo **cierra automáticamente** al salir (aunque haya un error). Aquí abre un *run* de MLflow y lo cierra solo al terminar el bloque. Es la forma segura y recomendada.

Dentro del run:
- **`mlflow.log_artifacts(local_dir=model_path, artifact_path="BasicChatBot")`** sube los ficheros del modelo (que guardamos antes en disco) al almacén de artefactos del run.
- Guardamos `run.info.run_id` porque lo necesitaremos en el siguiente paso para registrar el modelo.

### 📊 Qué más puedes registrar en un run

| Función | Qué registra |
|---------|--------------|
| `mlflow.log_param("temp", 0.7)` | Un **parámetro** de configuración |
| `mlflow.log_metric("latencia", 1.2)` | Una **métrica** numérica (se puede graficar) |
| `mlflow.log_artifact("archivo.txt")` | Un **fichero** suelto |
| `mlflow.log_artifacts("carpeta/")` | Una **carpeta** entera |
| `mlflow.pyfunc.log_model(...)` | Un **modelo** completo (lo más habitual) |

> 👀 Todo esto aparece en la pestaña **Experiments** de la interfaz de Databricks, donde puedes comparar runs lado a lado.

In [ ]:
import mlflow

run_id = None

# Log the model as an artifact
with mlflow.start_run() as run:
    mlflow.log_artifacts(local_dir=model_path, artifact_path="BasicChatBot")
    print(f"Model logged with run ID: {run.info.run_id}")
    run_id = run.info.run_id

## 🔟 Registrando el modelo en Unity Catalog (`register_model`)

### 📚 Diferencia clave: *log* vs *register* (¡muy preguntado en la certificación!)

- **Loguear (`log`)** un modelo = guardarlo asociado a un *run* (un experimento). Es como un "borrador" de un intento concreto.
- **Registrar (`register`)** un modelo = darlo de alta en el **Model Registry de Unity Catalog**, con un **nombre oficial y versiones** (v1, v2, v3...). Es el paso que lo convierte en un activo gobernado, compartible y desplegable.

### 🏛️ Nombres en Unity Catalog: tres niveles

En Unity Catalog **todo** se nombra con tres niveles: `catalogo.esquema.objeto`. Por eso, en un proyecto real, el nombre del modelo debería ser algo como:

```python
mlflow.register_model(f"runs:/{run_id}/BasicChatBot", "mi_catalogo.mi_esquema.BasicChatBot")
```

> ⚠️ En este notebook se usa solo `"BasicChatBot"` (sin catálogo ni esquema) por simplicidad. Para que funcione contra Unity Catalog hay que: 1) ejecutar `mlflow.set_registry_uri("databricks-uc")` y 2) usar el nombre de tres niveles.

### 🔗 La URI `runs:/{run_id}/BasicChatBot`

Esa cadena le dice a MLflow **de dónde** sacar el modelo a registrar:
- `runs:/` → desde un run.
- `{run_id}` → el identificador del run que guardamos antes.
- `/BasicChatBot` → la ruta del artefacto dentro del run.

### 🏷️ Versiones y Alias (sustituyen a los antiguos "Stages")

En el Model Registry **de Unity Catalog** ya no se usan los *stages* "Staging/Production". Ahora se usan **alias** (etiquetas que apuntan a una versión), por ejemplo:
- `@champion` → la versión en producción.
- `@challenger` → la versión candidata que se está evaluando.

```python
from mlflow import MlflowClient
MlflowClient().set_registered_model_alias("mi_catalogo.mi_esquema.BasicChatBot", "champion", version=1)
```

> 💡 **Para la certificación**: recuerda que en UC se usan **aliases + tags** en lugar de los stages clásicos del antiguo Workspace Model Registry.

In [ ]:
mlflow.register_model(f"runs:/{run_id}/BasicChatBot", "BasicChatBot")

## 1️⃣1️⃣ Desplegando y probando el endpoint en tiempo real

Una vez registrado el modelo en Unity Catalog, lo **desplegamos** creando un **Serving Endpoint** (una API REST en tiempo real). Esto se puede hacer desde la **interfaz** o por **código**.

### 🖱️ Opción A — Desde la interfaz (UI)

1. Ve a **Serving** (menú lateral izquierdo) → **Create serving endpoint**.
2. Dale un nombre al endpoint.
3. En *Entity*, elige tu modelo registrado y la **versión** (o un alias como `@champion`).
4. Elige el **tamaño de cómputo** (*Compute Type*: Small / Medium / Large; CPU o GPU).
5. **Activa "Scale to zero"** si quieres ahorrar (ver explicación abajo).
6. *Create*. El endpoint tarda unos minutos en quedar **Ready** (verde).

### 💻 Opción B — Desde código (recomendado y reproducible)

```python
from mlflow.deployments import get_deploy_client

client = get_deploy_client("databricks")
client.create_endpoint(
    name="basicchatbot-endpoint",
    config={
        "served_entities": [{
            "entity_name": "mi_catalogo.mi_esquema.BasicChatBot",
            "entity_version": "1",
            "workload_size": "Small",      # Small / Medium / Large
            "scale_to_zero_enabled": True   # 👈 escala a cero
        }]
    }
)
```

### 🧪 Probar el endpoint

Desde la interfaz, en la pestaña del endpoint hay un botón **Query endpoint** donde puedes pegar la entrada de prueba (la celda JSON de abajo). El formato `dataframe_split` es el estándar de MLflow para enviar un `DataFrame` por HTTP:
- `columns`: los nombres de las columnas (deben coincidir con la firma: `user_query`).
- `data`: las filas de valores.

Por código se consulta así:

```python
client.predict(
    endpoint="basicchatbot-endpoint",
    inputs={"dataframe_split": {"columns": ["user_query"], "data": [["how is Gotham today?"]]}}
)
```

---

### ⚡ ¿Qué es "Scale to Zero"? (pregunta clásica de certificación)

**Scale to zero** ("escalar a cero") significa que, si el endpoint **no recibe peticiones durante un rato** (por defecto, ~30 minutos), Databricks **apaga el cómputo automáticamente** y deja de cobrarte por él. Cuando llega una nueva petición, **se vuelve a encender solo**.

| | Scale to Zero **activado** | Scale to Zero **desactivado** |
|---|---|---|
| **Coste** | 💰 Bajo: solo pagas cuando se usa | 💰💰 Mayor: pagas siempre, esté o no en uso |
| **Latencia 1ª petición** | ⏳ *Cold start*: la primera petición tras el apagado tarda más (segundos) mientras arranca | ⚡ Siempre rápido, sin arranques en frío |
| **Cuándo usarlo** | Desarrollo, demos, tráfico esporádico o impredecible | Producción con tráfico constante y baja latencia exigida |

> 🧊 **Cold start** = "arranque en frío": el retraso de la **primera** petición después de que el endpoint estuvo apagado. Las siguientes peticiones ya van rápidas.

### 📈 Otras opciones de Model Serving que conviene conocer

| Opción | Para qué sirve |
|--------|----------------|
| **Workload size** (Small/Medium/Large) | Capacidad de cómputo; afecta a cuántas peticiones simultáneas aguanta. |
| **CPU vs GPU** | GPU para modelos pesados (LLMs propios); CPU para lógica ligera como esta. |
| **Auto-scaling** | El endpoint añade/quita réplicas según la carga de tráfico. |
| **Traffic split (A/B)** | Repartir el tráfico entre varias versiones (p. ej. 90% v1, 10% v2) para probar sin riesgo. |
| **Inference Tables** | Guardan automáticamente todas las peticiones y respuestas en una tabla de UC, para auditoría y monitorización. |
| **AI Gateway** | Capa de control: límites de uso (*rate limits*), control de coste, registro de uso, seguridad de payloads. |
| **Provisioned throughput** | Rendimiento garantizado y reservado para modelos fundacionales (en vez de pay-per-token). |

---

## 🎓 Resumen del flujo completo (chuleta para la certificación)

```
Instalar libs → reiniciar Python → crear cliente → probar LLM
   → escribir clase PythonModel (predict) → save_model → load_model → probar
   → log (run de MLflow) → register_model (Unity Catalog, versiones/alias)
   → deploy (Serving Endpoint, scale-to-zero) → query
```

| Verbo MLflow | Qué hace | Resultado |
|--------------|----------|-----------|
| **save** | Guarda el modelo en una carpeta local | Artefactos en disco |
| **load** | Reconstruye el modelo desde disco | Objeto `pyfunc` para predecir |
| **log** | Asocia el modelo a un *run* (experimento) | Trazabilidad en *Experiments* |
| **register** | Lo da de alta en el Model Registry de UC | Nombre oficial + versiones |
| **deploy** | Crea un Serving Endpoint | API REST en tiempo real |

> 🧠 **Conceptos que suelen caer**: diferencia log vs register; firma del modelo (`signature`); nombres de tres niveles en UC; alias `@champion`/`@challenger` (no stages); scale to zero y cold start; Foundation Model APIs (pay-per-token) vs custom model serving.

In [ ]:
{
  "dataframe_split": {
    "columns": [
      "user_query"
    ],
    "data": [
      [
        "how is Gotham City doing today?"
      ]
    ]
  }
}